# PredictX · FD001 exploration
Explore the fitting engines, inspect the RUL labels, and look for sensor patterns.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if not (root / "src" / "data_loader.py").is_file():
    raise RuntimeError("Launch Jupyter from the project-X folder.")
sys.path.insert(0, str(root))

from src.data_loader import RAW_DIR, SENSORS, load_trajectories
from src.eda import FIGURE_DIR, generate_eda
from src.preprocessing import add_rul, split_engines

full_train = load_trajectories(RAW_DIR / "train_FD001.txt")
train, validation = split_engines(add_rul(full_train))
print(f"{train.unit_id.nunique()} fitting engines and {validation.unit_id.nunique()} validation engines")
assert set(train.unit_id).isdisjoint(validation.unit_id)
train[["unit_id", "cycle", "RUL"]].head()

## Sensor variability
Sensors with a single distinct value have zero variance in the fitting partition. The heatmap covers varying sensors.

In [ ]:
sensor_summary = train[SENSORS].describe().T
sensor_summary["distinct_values"] = train[SENSORS].nunique()
sensor_summary.round(4)

In [ ]:
import matplotlib.pyplot as plt

varying = sensor_summary.index[sensor_summary["distinct_values"] > 1]
correlations = train[varying].corr()
fig, ax = plt.subplots(figsize=(9, 8), layout="constrained")
heatmap = ax.imshow(correlations, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(varying)), labels=varying, rotation=60, ha="right")
ax.set_yticks(range(len(varying)), labels=varying)
ax.set_title("Sensor correlations in fitting engines")
fig.colorbar(heatmap, ax=ax, label="Pearson correlation")
plt.show()

## Engine lifetimes and trajectories
Compare engine lifetimes, the RUL countdown, and the trajectory of sensor 11.

In [ ]:
from IPython.display import Image, display

generate_eda()
for name in ("engine_lifetimes.png", "rul_trajectories.png", "sensor_11_trajectories.png"):
    display(Image(filename=str(FIGURE_DIR / name)))